## Phase 1: Library Imports and Data Cleaning
Load the dataset and transform the list of strings into a One-Hot matrix for the Apriori algorithm.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')

CLUSTER_NAMES_MAP = {
    0: "The Plant-Based Household",
    1: "The Family Budget Optimizer",
    2: "The Promotion-Driven Pet Parent",
    3: "The Demanding Premium Tech Consumer",
    4: "The Wellness Customer",
    5: "The Affluent Health-Conscious Buyer"
}


def load_and_prepare_data(basket_path='customer_basket (1).csv', clusters_path='dataset_clusters.csv'):
    basket = pd.read_csv(basket_path)
    clusters = pd.read_csv(clusters_path)
    clusters['cluster'] = clusters['cluster'].map(CLUSTER_NAMES_MAP)
    return pd.merge(clusters, basket, on='customer_id', how='inner')


def convert_string_to_list(string):
    return json.loads(string.replace("'", '"'))


def calculate_metric_range(rules_df, metric='lift'):
    if rules_df.empty:
        return f"{metric} Range: N/A (No rules generated)"
    min_val = rules_df[metric].min()
    max_val = rules_df[metric].max()
    return f"{metric} Range: {max_val - min_val:.4f} (Min: {min_val:.4f}, Max: {max_val:.4f})"


def plot_top_products(basket_list, title_suffix=""):
    te = TransactionEncoder()
    te_ary = te.fit(basket_list).transform(basket_list)
    df_trans = pd.DataFrame(te_ary, columns=te.columns_)
    
    top_items = df_trans.sum().sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(10, 5))
    top_items.plot(kind='bar', color='#5c6bc0', edgecolor='black')
    plt.title(f'Top 10 Most Frequent Products - {title_suffix}', fontweight='bold')
    plt.ylabel('Absolute Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    filename = f"top_products_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(filename)
    plt.close()


def plot_association_results(rules_df, title_suffix=""):
    if rules_df.empty:
        return
    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(rules_df['support'], rules_df['confidence'], 
                          c=rules_df['lift'], cmap='viridis', alpha=0.6)
    plt.colorbar(scatter, label='Lift')
    plt.title(f'Rules Scatter Plot (Support vs Confidence) - {title_suffix}', fontweight='bold')
    plt.xlabel('Support')
    plt.ylabel('Confidence')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    filename = f"rules_scatter_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(filename)
    plt.close()


def run_association_rules(basket_data, min_support=0.05, min_confidence=0.2, split_train=True):
    if split_train:
        train_size = int(len(basket_data) * 0.8)
        data_to_model = basket_data[:train_size]
    else:
        data_to_model = basket_data
        
    te = TransactionEncoder()
    te_fit = te.fit(data_to_model).transform(data_to_model)
    transactions_items = pd.DataFrame(te_fit, columns=te.columns_)
    
    frequent_itemsets = apriori(transactions_items, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        return frequent_itemsets, pd.DataFrame()
        
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
    rules = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)
    return frequent_itemsets, rules


if __name__ == "__main__":
    try:
        df_basket_clusters = load_and_prepare_data(
            basket_path='customer_basket (1).csv', 
            clusters_path='dataset_clusters.csv'
        )
        df_basket_clusters['list_of_goods_parsed'] = df_basket_clusters['list_of_goods'].apply(convert_string_to_list)
        
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        pd.set_option('display.max_colwidth', None)

        def clean_rules_df(df_rules):
            if df_rules.empty:
                return df_rules
            df_cool = df_rules.copy()
            df_cool['antecedents'] = df_cool['antecedents'].apply(lambda x: f"[{', '.join(list(x))}]")
            df_cool['consequents'] = df_cool['consequents'].apply(lambda x: f"[{', '.join(list(x))}]")
            metrics = ['support', 'confidence', 'lift', 'leverage', 'conviction', 'zhangs_metric']
            for m in metrics:
                if m in df_cool.columns:
                    df_cool[m] = df_cool[m].round(4)
            return df_cool

        def clean_items_df(df_items):
            if df_items.empty:
                return df_items
            df_cool = df_items.copy()
            df_cool['itemsets'] = df_cool['itemsets'].apply(lambda x: f"[{', '.join(list(x))}]")
            df_cool['support'] = df_cool['support'].round(4)
            return df_cool

        
        # PART 1: GLOBAL RULES (ALL CUSTOMERS)

        print("\n" + "="*80)
        print("=== PART 1: GLOBAL ASSOCIATION RULES (ALL CUSTOMERS COMBINED) ===")
        print("="*80)
        all_baskets = df_basket_clusters['list_of_goods_parsed'].tolist()
        plot_top_products(all_baskets, "Global Dataset")
        
        items_all, rules_all = run_association_rules(all_baskets, min_support=0.02, min_confidence=0.15, split_train=True)
        
        print(f"\n-> [Global] Frequent Itemsets (Top 10):")
        display(clean_items_df(items_all.head(10)))
        
        print(f"\n-> [Global] Association Rules Generated (Top 15 sorted by Lift):")
        if not rules_all.empty:
            df_all_view = rules_all[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15)
            display(clean_rules_df(df_all_view))
            print(f"\n{calculate_metric_range(rules_all, 'lift')}")
            plot_association_results(rules_all, "Global Dataset")
        else:
            print("No rules generated globally for the current threshold limits.")

        # PART 2: STANDARDIZED ANALYSIS PER CLUSTER (AUTOMATED LOOP)

        print("\n" + "="*80)
        print("=== PART 2: STANDARDIZED ASSOCIATION RULES ANALYSIS BY CLUSTER COEFFICIENT ===")
        print("="*80)
        
        DEFAULT_SUPPORT = 0.02
        DEFAULT_CONFIDENCE = 0.15
        
        for cluster_id, cluster_name in CLUSTER_NAMES_MAP.items():
            print("\n" + "-"*60)
            print(f"ANALYSIS FOR CLUSTER {cluster_id}: {cluster_name}")
            print("-"*60)
            
            df_sub_cluster = df_basket_clusters[df_basket_clusters['cluster'] == cluster_name]
            cluster_baskets = df_sub_cluster['list_of_goods_parsed'].tolist()
            
            print(f"Total transactions within this segment: {len(cluster_baskets)}")
            
            if len(cluster_baskets) > 0:
                plot_top_products(cluster_baskets, cluster_name)
                
                items_cl, rules_cl = run_association_rules(
                    cluster_baskets, 
                    min_support=DEFAULT_SUPPORT, 
                    min_confidence=DEFAULT_CONFIDENCE, 
                    split_train=True
                )
                
                print(f"\n   -> Frequent Itemsets Found (Top 10 Sample out of {len(items_cl)}):")
                if not items_cl.empty:
                    display(clean_items_df(items_cl.head(10)))
                else:
                    print(f"      No frequent itemsets passed the {DEFAULT_SUPPORT*100}% support threshold.")
                
                print(f"\n   -> Association Rules Found (Top 10 sorted by Lift out of {len(rules_cl)}):")
                if not rules_cl.empty:
                    df_cl_view = rules_cl[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)
                    display(clean_rules_df(df_cl_view))
                    print(f"\n      {calculate_metric_range(rules_cl, 'lift')}")
                    plot_association_results(rules_cl, cluster_name)
                else:
                    print(f"      No rules generated for the thresholds (Support={DEFAULT_SUPPORT*100}%, Confidence={DEFAULT_CONFIDENCE*100}%).\n"
                          f"      Indicates independent purchasing behavior for products within this segment.")
            else:
                print(f"   -> Alert: No records found in the dataset for this cluster.")

        print("\n" + "="*80)
        print("=== DATA PROCESSING COMPLETED SUCCESSFULLY ===")
        print("="*80)

    except Exception as e:
        print(f"\n[Execution Error]: {e}")


=== PART 1: GLOBAL ASSOCIATION RULES (ALL CUSTOMERS COMBINED) ===

-> [Global] Frequent Itemsets (Top 10):


,support,itemsets
0,0.1203,[airpods]
1,0.0632,[almonds]
2,0.0698,[antioxydant juice]
3,0.1274,[asparagus]
4,0.0664,[avocado]
5,0.0844,[babies food]
6,0.0688,[bacon]
7,0.0510,[barbecue sauce]
8,0.0558,[beer]
9,0.0452,[black beer]



-> [Global] Association Rules Generated (Top 15 sorted by Lift):


,antecedents,consequents,support,confidence,lift
0,[airpods],[energy drink],0.0248,0.2058,2.8029
1,[energy drink],[airpods],0.0248,0.3372,2.8029
2,[bluetooth headphones],[airpods],0.0241,0.3355,2.7887
3,[airpods],[bluetooth headphones],0.0241,0.2000,2.7887
4,[cereals],[eggs],0.0224,0.2213,2.3772
5,[eggs],[cereals],0.0224,0.2408,2.3772
6,[butter],[eggs],0.0206,0.2136,2.2941
7,[eggs],[butter],0.0206,0.2216,2.2941
8,[fresh bread],[cereals],0.0228,0.2286,2.2568
9,[cereals],[fresh bread],0.0228,0.2246,2.2568



lift Range: 0.7120 (Min: 2.0909, Max: 2.8029)

=== PART 2: STANDARDIZED ASSOCIATION RULES ANALYSIS BY CLUSTER COEFFICIENT ===

------------------------------------------------------------
ANALYSIS FOR CLUSTER 0: The Plant-Based Household
------------------------------------------------------------
Total transactions within this segment: 10080

   -> Frequent Itemsets Found (Top 10 Sample out of 271):


...


   -> Association Rules Found (Top 10 sorted by Lift out of 0):
      No rules generated for the thresholds (Support=2.0%, Confidence=15.0%).
      Indicates independent purchasing behavior for products within this segment.

------------------------------------------------------------
ANALYSIS FOR CLUSTER 5: The Affluent Health-Conscious Buyer
------------------------------------------------------------
Total transactions within this segment: 4832

   -> Frequent Itemsets Found (Top 10 Sample out of 267):


,support,itemsets
0,0.0655,[airpods]
1,0.0461,[almonds]
2,0.0528,[antioxydant juice]
3,0.0893,[asparagus]
4,0.0422,[avocado]
5,0.2435,[babies food]
6,0.0411,[bacon]
7,0.0507,[barbecue sauce]
8,0.0507,[beer]
9,0.0463,[black beer]



   -> Association Rules Found (Top 10 sorted by Lift out of 164):


,antecedents,consequents,support,confidence,lift
0,"[babies food, napkins]",[cooking oil],0.0318,0.4155,1.8439
1,"[babies food, cooking oil]",[napkins],0.0318,0.3880,1.7120
2,"[cooking oil, dog food]",[babies food],0.0298,0.4137,1.6991
3,"[cooking oil, napkins]",[babies food],0.0318,0.4128,1.6953
4,"[babies food, dog food]",[cooking oil],0.0298,0.3734,1.6568
5,"[cooking oil, dog food]",[napkins],0.0261,0.3633,1.6030
6,"[napkins, dog food]",[cooking oil],0.0261,0.3495,1.5508
7,"[babies food, cooking oil]",[dog food],0.0298,0.3628,1.5459
8,[milk],[napkins],0.0492,0.3442,1.5187
9,[napkins],[milk],0.0492,0.2169,1.5187



      lift Range: 0.8252 (Min: 1.0188, Max: 1.8439)

=== DATA PROCESSING COMPLETED SUCCESSFULLY ===


## Main Marketing Insights & Actions

### Insight A: Global Store Drivers (Traffic Magnets)
* **Finding:** Across the entire store, items like `asparagus`, `airpods`, and `cereals` dominate absolute purchasing frequencies. 
* **Marketing Action:** Treat these as **"Loss-Leaders"** in mass-media campaigns or catalog covers. High-reach discounts on these specific items will efficiently drive overall traffic to the checkout.

### Insight B: Cluster 4 — "The Wellness Customer" (High-Conversion Target)
* **Finding:** This segment yielded the most robust patterns, generating **14 strong association rules** with a **Maximum Lift of 2.87**. Customers are nearly 3 times more likely to buy a secondary healthy item if promoted next to the cluster's anchor item (`asparagus`).
* **Marketing Action — "Live Well" Campaign:** Launch immediate product bundles or targeted discount coupons pairing `asparagus` with its linked wellness items. Trigger automated "Frequently Bought Together" carousels at the digital checkout for this cluster.

### Insight C: Cluster 3 — "The Demanding Premium Tech Consumer" (Anti-Bundling Strategy)
* **Finding:** Cluster 3 generated **no meaningful association rules**. Purchases within this segment are statistically independent (buying Tech Product A does not influence buying Tech Product B).
* **Marketing Action — Individual Margin Strategy:** **Do not waste budget** creating product bundles or multi-pack discounts for this group. Pivot exclusively to **Up-Selling** (premium upgrades of the same item) or brand-loyalty perks (e.g., Apple- or Samsung-specific campaigns).
